In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from scipy.stats import kendalltau
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler

BASE = Path(r"c:\projects\ml_learn\HackRush")
TRAIN_PATH = BASE / "Mysterious_Affair_at_Styles_Train_Data.csv"
BOOK_A_PATH = BASE / "BookA_test.csv"
BOOK_B_PATH = BASE / "BookB_test.csv"
BOOK_A_OUT = BASE / "BookA.csv"
BOOK_B_OUT = BASE / "BookB.csv"

train_df = pd.read_csv(TRAIN_PATH)
book_a_df = pd.read_csv(BOOK_A_PATH)
book_b_df = pd.read_csv(BOOK_B_PATH)

EMBED_MODEL = SentenceTransformer("all-MiniLM-L6-v2")


def clean_text(text: str) -> str:
    text = "" if pd.isna(text) else str(text)
    return re.sub(r"\s+", " ", text).strip()


def split_sentences(text: str) -> list[str]:
    text = clean_text(text)
    parts = re.split(r"(?<=[.!?])\s+", text)
    return [part.strip() for part in parts if part.strip()]


def head_snippet(text: str, n_sentences: int = 2, max_chars: int = 700) -> str:
    sentences = split_sentences(text)
    snippet = " ".join(sentences[:n_sentences]) if sentences else clean_text(text)
    return snippet[:max_chars]


def tail_snippet(text: str, n_sentences: int = 2, max_chars: int = 700) -> str:
    sentences = split_sentences(text)
    snippet = " ".join(sentences[-n_sentences:]) if sentences else clean_text(text)
    return snippet[-max_chars:]


def chapter_start_flag(text: str) -> int:
    return int(bool(re.match(r"(?i)^\s*(chapter|part)\b", clean_text(text))))


def encode_texts(texts: list[str]) -> np.ndarray:
    return EMBED_MODEL.encode(
        texts,
        batch_size=32,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )


def build_representations(frame: pd.DataFrame) -> dict[str, object]:
    texts = [clean_text(t) for t in frame["text"].tolist()]
    heads = [head_snippet(t) for t in texts]
    tails = [tail_snippet(t) for t in texts]

    rep = {
        "texts": texts,
        "heads": heads,
        "tails": tails,
        "len": np.array([len(t) for t in texts], dtype=float),
        "chapter": np.array([chapter_start_flag(t) for t in texts], dtype=float),
        "full_emb": encode_texts(texts),
        "head_emb": encode_texts(heads),
        "tail_emb": encode_texts(tails),
    }
    return rep


def page_feature_matrix(rep: dict[str, object]) -> np.ndarray:
    lengths = rep["len"].reshape(-1, 1)
    chapters = rep["chapter"].reshape(-1, 1)
    return np.hstack(
        [
            rep["full_emb"],
            rep["head_emb"],
            rep["tail_emb"],
            lengths / max(float(lengths.max()), 1.0),
            chapters,
        ]
    )


train_rep = build_representations(train_df)
book_a_rep = build_representations(book_a_df)
book_b_rep = build_representations(book_b_df)

all_features = np.vstack(
    [
        page_feature_matrix(train_rep),
        page_feature_matrix(book_a_rep),
        page_feature_matrix(book_b_rep),
    ]
)
scaler = StandardScaler()
scaled_all = scaler.fit_transform(all_features)

pca_dim = min(32, scaled_all.shape[1], scaled_all.shape[0] - 1)
pca = PCA(n_components=pca_dim, random_state=42)
pca.fit(scaled_all)

train_features = pca.transform(scaler.transform(page_feature_matrix(train_rep)))
book_a_features = pca.transform(scaler.transform(page_feature_matrix(book_a_rep)))
book_b_features = pca.transform(scaler.transform(page_feature_matrix(book_b_rep)))

train_target = np.linspace(0.0, 1.0, len(train_df), dtype=float)
position_model = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0])
position_model.fit(train_features, train_target)


def order_from_scores(scores: np.ndarray) -> list[int]:
    return list(np.argsort(scores, kind="mergesort"))


def save_submission(frame: pd.DataFrame, features: np.ndarray, out_path: Path) -> list[int]:
    scores = position_model.predict(features)
    order = order_from_scores(scores)
    submission = pd.DataFrame(
        {
            "original_page": np.arange(1, len(order) + 1, dtype=int),
            "shuffled_page": frame.iloc[order]["page"].to_numpy(dtype=int),
        }
    )
    submission.to_csv(out_path, index=False)
    return order


book_a_order = save_submission(book_a_df, book_a_features, BOOK_A_OUT)
book_b_order = save_submission(book_b_df, book_b_features, BOOK_B_OUT)


def kendall_on_self_shuffle(frame: pd.DataFrame, seed: int = 123) -> float:
    shuffled = frame.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    shuffled_rep = build_representations(shuffled)
    shuffled_features = pca.transform(scaler.transform(page_feature_matrix(shuffled_rep)))
    scores = position_model.predict(shuffled_features)
    order = order_from_scores(scores)
    predicted_positions = np.empty(len(shuffled), dtype=int)
    for rank, idx in enumerate(order, start=1):
        predicted_positions[idx] = rank
    true_positions = shuffled["page"].to_numpy(dtype=int)
    tau = kendalltau(true_positions, predicted_positions).correlation
    return float(tau)


train_tau = kendall_on_self_shuffle(train_df)
print(f"Train self-shuffle Kendall tau: {train_tau:.4f}")
print(f"BookA output rows: {len(book_a_order)} -> {BOOK_A_OUT.name}")
print(f"BookB output rows: {len(book_b_order)} -> {BOOK_B_OUT.name}")
print("Done.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6865.96it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
